In [1]:
# Cell 0 - Setup utilities for the notebook
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)
pd.set_option('display.width', 120)

def safe_display(df, n=5, name=None):
    """Display first n rows or a friendly message if df is None/empty."""
    if df is None:
        print(f"{name or 'DataFrame'} is None")
        return
    if getattr(df, "empty", False):
        print(f"{name or 'DataFrame'} is empty")
        return
    display(df.head(n))

def suggest_user_examples(top_n=1):
    """
    Return example userIds for categories if the related variables exist in the notebook.
    Looks for globals created in later cells: ratings, user_activity, user_mean_rating,
    user_genre_profile_normalized.
    """
    out = {}
    # Activity-based
    ua = globals().get('user_activity')
    if ua is None and 'ratings' in globals():
        try:
            ua = ratings.groupby('userId')['rating'].count()
        except Exception:
            ua = None
    if ua is not None:
        out['power_users'] = ua.sort_values(ascending=False).head(top_n).index.tolist()
        out['cold_start'] = ua.sort_values(ascending=True).head(top_n).index.tolist()

    # Rating-behavior
    umr = globals().get('user_mean_rating')
    if umr is None and 'ratings' in globals():
        try:
            umr = ratings.groupby('userId')['rating'].mean()
        except Exception:
            umr = None
    if umr is not None:
        out['critical_users'] = umr.sort_values(ascending=True).head(top_n).index.tolist()
        out['easy_users'] = umr.sort_values(ascending=False).head(top_n).index.tolist()

    # Genre-based (expects user_genre_profile_normalized)
    ug = globals().get('user_genre_profile_normalized')
    if ug is not None:
        try:
            niche = ug[ug.max(axis=1) > 0.8]
            diverse = ug[ug.max(axis=1) < 0.3]
            out['niche'] = niche.head(top_n).index.tolist()
            out['diverse'] = diverse.head(top_n).index.tolist()
            if 'Sci-Fi' in ug.columns:
                out['scifi_fans'] = ug[ug['Sci-Fi'] > 0.7].head(top_n).index.tolist()
        except Exception:
            pass

    return out